# 00 — Fundamentos: de C para Python

Quatro ideias, na ordem:

1. Variáveis sem declaração de tipo
2. Arrays do numpy e o fim do laço `for` (vetorização)
3. Objetos: uma `struct` que tem funções dentro
4. DataFrame: uma tabela de structs

Rode uma célula por vez (**Shift+Enter**). Nas células marcadas **✎ Tente você**, mude o código e rode de novo. Pode quebrar à vontade: um erro no notebook não estraga nada.

## 1. Variáveis sem declaração

Em C:
```c
double g = 978032.67;
int n = 2816;
```
Em Python, você só atribui. O tipo vem do valor.

In [2]:
g = 978032.67
n = 2816
nome = "CPRM"

print(type(g), type(n), type(nome))

<class 'float'> <class 'int'> <class 'str'>


`type(x)` diz de que tipo é uma variável. Use sempre que tiver dúvida.

A mesma variável pode mudar de tipo (em C isso seria erro de compilação):

In [3]:
x = 10
print(type(x))
x = "agora sou texto"
print(type(x))

<class 'int'>
<class 'str'>


## 2. Arrays do numpy: sem laço `for`

Em C, para calcular a anomalia ar-livre de 5 estações você escreveria:
```c
for (i = 0; i < 5; i++)
    fa[i] = g_obs[i] - gamma[i] + 0.3086 * h[i];
```
Com numpy, a operação vale para o array inteiro de uma vez:

In [4]:
import numpy as np

g_obs = np.array([978600.1, 978550.3, 978720.8, 978480.0, 978650.2])   # mGal
gamma = np.array([978750.0, 978752.1, 978748.3, 978755.6, 978751.0])  # mGal
h     = np.array([520.0, 810.0, 150.0, 1020.0, 430.0])                 # m

fa = g_obs - gamma + 0.3086 * h
print(fa)

[10.572 48.166 18.79  39.172 31.898]


Não há laço, mas ele existe: o numpy roda o laço **em C, por baixo**. Isso se chama **vetorização**.

Por que importa? Velocidade. Compare os dois jeitos com 1 milhão de valores:

In [5]:
import time

N = 1_000_000
a = np.random.rand(N)
b = np.random.rand(N)

# jeito C, laço em Python
t0 = time.time()
c = np.empty(N)
for i in range(N):
    c[i] = a[i] - b[i]
t_laco = time.time() - t0

# jeito numpy
t0 = time.time()
c2 = a - b
t_vet = time.time() - t0

print(f"laço: {t_laco:.3f} s   vetorizado: {t_vet:.4f} s   ({t_laco / t_vet:.0f}x mais rápido)")

laço: 0.495 s   vetorizado: 0.0024 s   (209x mais rápido)


Regra prática: **se você está escrevendo um `for` sobre os elementos de um array, provavelmente existe um jeito numpy de fazer sem ele.**

Outras operações úteis, todas sem laço:

In [6]:
print("média:", fa.mean())
print("máximo:", fa.max())
print("posição do máximo:", fa.argmax())   # índice começa em 0, como em C
print("maiores que 30:", fa > 30)          # array de verdadeiro/falso
print("só os maiores que 30:", fa[fa > 30])  # usa o verdadeiro/falso como filtro

média: 29.719600000004647
máximo: 48.16600000006983
posição do máximo: 1
maiores que 30: [False  True False  True  True]
só os maiores que 30: [48.166 39.172 31.898]


A última linha é muito usada: `fa[fa > 30]` significa "os elementos de `fa` onde a condição é verdadeira". Em C seria um laço com `if` e um segundo array.

**✎ Tente você:** mostre só as altitudes `h` acima de 500 m. Depois, a média de `fa` só dessas estações.

In [ ]:
# escreva aqui


## 3. Objetos: uma struct com funções dentro

Em C:
```c
struct Estacao { char nome[20]; double g_obs, lat, h; };
double ar_livre(struct Estacao *e);   // função separada
```
Em Python, a definição da struct se chama **classe**. As funções podem morar dentro dela e se chamam **métodos**:

In [7]:
class Estacao:
    def __init__(self, nome, g_obs, lat, h):
        # __init__ roda quando o objeto é criado: preenche os campos
        self.nome = nome
        self.g_obs = g_obs
        self.lat = lat
        self.h = h

    def gamma(self):
        # gravidade normal GRS80 (Somigliana)
        s2 = np.sin(np.radians(self.lat)) ** 2
        return 978032.67715 * (1 + 0.001931851353 * s2) / np.sqrt(1 - 0.0066943800229 * s2)

    def ar_livre(self):
        return self.g_obs - self.gamma() + 0.3086 * self.h

- `self` é o próprio objeto, o equivalente ao ponteiro `e` que a função recebia em C.
- Em C: `e->h`. Em Python: `self.h`.

Agora, criar e usar um objeto:

In [9]:
bh = Estacao("Belo Horizonte B", 978376.52, -19.87, 915.0)

print(bh.nome)          # campo
print(bh.h)             # campo
print(bh.ar_livre())    # método: repare nos parênteses

Belo Horizonte B
915.0
29.45250870013234


In [10]:
print(bh.nome)

Belo Horizonte B


O ponto (`.`) acessa tudo o que está dentro do objeto, seja campo, seja método. Método precisa de `()`, porque é uma função sendo chamada.

**✎ Tente você:** crie outra estação com dados inventados e calcule o ar-livre. Depois digite `bh.` numa célula nova e espere: o PyCharm lista tudo o que existe dentro do objeto.

In [ ]:
# escreva aqui


**Não é preciso escrever classes para fazer data science.** Quase sempre você só *usa* objetos que outras bibliotecas criaram. Entender a ideia acima basta para ler o código.

## 4. DataFrame: uma tabela de structs

Uma estação é um objeto. E 2.816 estações? Para isso existe o **DataFrame** do pandas: uma tabela em que cada linha é um registro e cada coluna é um array numpy com nome.

In [11]:
import pandas as pd

est = pd.DataFrame({
    "nome":  ["BH B", "JF B", "Santos D", "Barbacena B", "Vitória A"],
    "g_obs": g_obs,
    "h":     h,
    "fa":    fa,
})
est

,nome,g_obs,h,fa
0,BH B,978600.1,520.0,10.572
1,JF B,978550.3,810.0,48.166
2,Santos D,978720.8,150.0,18.790
3,Barbacena B,978480.0,1020.0,39.172
4,Vitória A,978650.2,430.0,31.898


Cada coluna se comporta como um array numpy: opera sem laço.

In [12]:
est["fa_por_km"] = est["fa"] / (est["h"] / 1000)
est

,nome,g_obs,h,fa,fa_por_km
0,BH B,978600.1,520.0,10.572,20.330769
1,JF B,978550.3,810.0,48.166,59.464198
2,Santos D,978720.8,150.0,18.790,125.266667
3,Barbacena B,978480.0,1020.0,39.172,38.403922
4,Vitória A,978650.2,430.0,31.898,74.181395


Filtrar linhas funciona como no numpy, com uma condição entre colchetes:

In [13]:
est[est["h"] > 500]

,nome,g_obs,h,fa,fa_por_km
0,BH B,978600.1,520.0,10.572,20.330769
1,JF B,978550.3,810.0,48.166,59.464198
3,Barbacena B,978480.0,1020.0,39.172,38.403922


E o próprio DataFrame é um objeto, com muitos métodos:

In [14]:
est.describe().round(1)

,g_obs,h,fa,fa_por_km
count,5.0,5.0,5.0,5.0
mean,978600.3,586.0,29.7,63.5
std,92.2,338.1,15.2,40.1
min,978480.0,150.0,10.6,20.3
25%,978550.3,430.0,18.8,38.4
50%,978600.1,520.0,31.9,59.5
75%,978650.2,810.0,39.2,74.2
max,978720.8,1020.0,48.2,125.3


In [15]:
est.sort_values("fa")

,nome,g_obs,h,fa,fa_por_km
0,BH B,978600.1,520.0,10.572,20.330769
2,Santos D,978720.8,150.0,18.790,125.266667
4,Vitória A,978650.2,430.0,31.898,74.181395
3,Barbacena B,978480.0,1020.0,39.172,38.403922
1,JF B,978550.3,810.0,48.166,59.464198


**✎ Tente você:**
1. Mostre só as estações com `fa` negativo.
2. Descubra a média de `h` usando `est["h"].mean()`.
3. Digite `est.` e explore a lista de métodos.

In [18]:
est[est["fa"] < 0]

,nome,g_obs,h,fa,fa_por_km


In [20]:
est["h"].mean()


np.float64(586.0)

In [21]:
print(est["h"].mean())

586.0


In [25]:
est.median(numeric_only=True)

g_obs        978600.100000
h               520.000000
fa               31.898000
fa_por_km        59.464198
dtype: float64

## Resumo

| C | Python |
|---|---|
| `double x = 1.0;` | `x = 1.0` |
| `for` sobre o array | operação no array inteiro (numpy) |
| `struct` + funções separadas | classe (campos + métodos) |
| `e->h` | `e.h` |
| array de `struct` | DataFrame |

Com isso, releia o notebook 01. `sandwell.lats`, `obs.groupby(...)` e `ax.set_title(...)` devem fazer mais sentido agora.